# 13 · Temporal Convolutional Networks (TCN) for Probabilistic Forecasting

So far every model was **local**: one model per series (SARIMA, GLM, quantile regression). Deep-learning forecasters
are usually **global**: *one* network is trained on windows from *many* related series. It learns shared patterns
(seasonal shapes, launch ramps, how demand reacts to its own recent past) and applies them to every series, including
series with very little history.

A **Temporal Convolutional Network** (Bai, Kolter & Koltun 2018, building on WaveNet, van den Oord et al. 2016) is the
convolutional way to read a sequence. This notebook builds one from scratch in PyTorch, gives it a **multi-horizon
quantile head**, and scores it against local baselines on a panel of 300 monthly SKU series.

> **Requirements:** this notebook needs PyTorch (`pip install torch`; the CPU build is enough). It runs in about
> two minutes on a laptop CPU.

## Theory

### 1. Causal convolutions
A 1-D convolution with kernel size $k$ slides a filter $w_0,\dots,w_{k-1}$ along the series. A **causal** convolution
only looks backwards, so the output at time $t$ never uses the future:

$$y_t=\sum_{i=0}^{k-1}w_i\,x_{t-d\cdot i}$$

In code: pad $(k-1)\,d$ zeros on the **left** only (or pad both sides and chop the right end, as PyTorch code often does).

### 2. Dilations: an exponentially growing memory
$d$ is the **dilation**, the gap between the inputs a filter sees. Stacking layers with $d=1,2,4,8,\dots$ makes the
receptive field grow **exponentially** with depth while the number of weights grows only **linearly**:

```
layer 4 (d=8)  ●───────────────●───────────────●
layer 3 (d=4)  ●───────●───────●───────●───────●
layer 2 (d=2)  ●───●───●───●───●───●───●───●───●
layer 1 (d=1)  ●─●─●─●─●─●─●─●─●─●─●─●─●─●─●─●─●
input          x x x x x x x x x x x x x x x x x  → y_t
```

With $L$ residual blocks, two convolutions per block and $d_\ell=2^{\ell}$, the **receptive field** (how many past steps
can influence $y_t$) is

$$R = 1 + 2(k-1)\,(2^{L}-1).$$

$k=3,\ L=4$ gives $R=61$ months. Rule of thumb: $R$ should cover the context window you feed in, and certainly at
least one seasonal cycle (12).

### 3. Residual blocks
```
x ──► [dilated causal conv → weight-norm → ReLU → dropout] ×2 ──► (+) ──► ReLU ──► out
│                                                                  ▲
└──────────────────── 1×1 conv (only if channel counts differ) ────┘
```
The skip connection makes deep stacks trainable (gradients flow through the identity path). **Weight normalisation**
and **dropout** are the regularisers used in the original TCN paper.

### 4. TCN vs the alternatives
| | RNN / LSTM (DeepAR, nb 14) | **TCN** | Transformer |
|---|---|---|---|
| Training | sequential in time, slower | **fully parallel** | parallel |
| Memory | unbounded in principle, short in practice | **fixed**: exactly $R$ steps | the whole context window |
| Gradients | can vanish / explode | stable (residual, no recurrence) | stable |
| Inductive bias | a running state | **local patterns at multiple scales** | arbitrary pairwise links |

### 5. From encoder to *probabilistic* forecast
The TCN only produces a feature vector $\mathbf h_t$ summarising the context. The **output head** turns it into a forecast:

| Head | Loss | Output | Course link |
|---|---|---|---|
| **Multi-horizon quantiles** (this notebook, as in MQ-CNN, Wen et al. 2017) | pinball | $H\times K$ quantiles in one shot | nb 02 |
| Distribution parameters, e.g. NegBin $(\mu,\alpha)$ | negative log-likelihood | a full distribution per step | nb 01, 08, 14 |
| Point forecast + conformal calibration | MSE / MAE, then residual quantiles | intervals | nb 04 |
| MC dropout (dropout on at prediction time) | any | samples (epistemic part only) | nb 11 |

Our head is **direct** (all 12 horizons at once, no feeding forecasts back in) and **non-crossing by construction**:
it outputs the lowest quantile plus *positive* increments, $q_{\tau_1}=a_1,\ q_{\tau_{j}}=q_{\tau_{j-1}}+\text{softplus}(a_j)$,
so $q_{\tau_1}\le q_{\tau_2}\le\dots$ always holds (compare the crossing fix in nb 02).

### 6. What makes a *global* model work
* **Windows.** Every series is cut into (context, target) pairs: $C=36$ months in, $H=12$ months out.
* **Scaling.** Series range from under 1 to hundreds of units a month. Each window is divided by
  $\nu=1+\text{mean(context)}$, the network works on scale-free numbers, and the outputs are multiplied by $\nu$ again.
  $\log\nu$ is also passed to the head, because the *shape* of the distribution depends on the scale (small counts are
  relatively noisier and pile up at zero).
* **Covariates.** Calendar month (as $\sin/\cos$) and the **age** of the SKU (months since its first sale).
* **No refitting.** At forecast time the trained network simply reads the latest 36 months of each series.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd
import plotly.graph_objects as go
import pf_utils as pf
pf.set_style()
try:
    import torch, torch.nn as nn, torch.nn.functional as F
except ImportError as e:
    raise ImportError("This notebook needs PyTorch: pip install torch (the CPU build is enough)") from e
torch.manual_seed(13); rng = np.random.default_rng(13)
print("torch", torch.__version__)

torch 2.14.0


## Part A: the mechanics

### A1. A causal dilated convolution by hand
We implement $y_t=\sum_i w_i x_{t-d i}$ with a plain loop and check it against PyTorch's `Conv1d` with left padding.

In [2]:
def causal_conv_numpy(x, w, d):
    """y_t = sum_i w[i] * x[t - d*i], with zeros before the start of the series."""
    k = len(w)
    y = np.zeros_like(x)
    for t in range(len(x)):
        for i in range(k):
            if t - d * i >= 0:
                y[t] += w[i] * x[t - d * i]
    return y

x = rng.normal(size=20).astype(np.float32)
w = np.array([0.5, -1.0, 2.0], dtype=np.float32)       # w_0 multiplies x_t, w_2 multiplies x_{t-2d}
conv = nn.Conv1d(1, 1, kernel_size=3, dilation=4, bias=False)
conv.weight.data = torch.tensor(w[::-1].copy()).view(1, 1, 3)   # PyTorch's kernel runs oldest -> newest
y_torch = conv(F.pad(torch.tensor(x).view(1, 1, -1), (2 * 4, 0))).detach().numpy().ravel()   # pad LEFT by (k-1)d
print("max |numpy - torch| =", np.abs(causal_conv_numpy(x, w, 4) - y_torch).max())

max |numpy - torch| = 4.7683716e-07


### A2. Receptive field and causality, measured with gradients
Which inputs can influence the **last** output? Take a randomly initialised TCN, backpropagate from its last output,
and see which input positions get a non-zero gradient. We also check that changing *future* inputs never changes an
earlier output.

(The `TemporalBlock` and `TCN` classes defined here are the ones we train in Part C.)

In [3]:
class Chomp(nn.Module):
    """Drop the last n time steps: turns symmetric padding into LEFT-only (causal) padding."""
    def __init__(self, n):
        super().__init__(); self.n = n
    def forward(self, x):
        return x[..., :-self.n]

class TemporalBlock(nn.Module):
    def __init__(self, c_in, c_out, k, d, dropout):
        super().__init__()
        pad = (k - 1) * d
        wn = nn.utils.parametrizations.weight_norm
        self.net = nn.Sequential(
            wn(nn.Conv1d(c_in, c_out, k, padding=pad, dilation=d)), Chomp(pad), nn.ReLU(), nn.Dropout(dropout),
            wn(nn.Conv1d(c_out, c_out, k, padding=pad, dilation=d)), Chomp(pad), nn.ReLU(), nn.Dropout(dropout))
        self.skip = nn.Conv1d(c_in, c_out, 1) if c_in != c_out else nn.Identity()
    def forward(self, x):
        return F.relu(self.net(x) + self.skip(x))

def tcn_encoder(c_in, channels, k, n_blocks, dropout=0.0):
    return nn.Sequential(*[TemporalBlock(c_in if i == 0 else channels, channels, k, 2**i, dropout) for i in range(n_blocks)])

def receptive_field(k, n_blocks):
    return 1 + 2 * (k - 1) * (2**n_blocks - 1)

LEN, k = 130, 3
rows, influence = [], []
for L in range(1, 6):
    enc = tcn_encoder(1, 8, k, L)
    xin = torch.randn(1, 1, LEN, requires_grad=True)
    enc(xin)[0, :, -1].sum().backward()                       # d(last output) / d(every input)
    g = (xin.grad.abs().numpy().ravel() > 0).astype(float)
    influence.append(g[::-1])                                 # index 0 = lag 0 (the current month)
    rows.append({"blocks L": L, "dilations": [2**i for i in range(L)], "formula R": receptive_field(k, L),
                 "measured R": int(g.sum())})
display(pd.DataFrame(rows).set_index("blocks L"))

enc = tcn_encoder(1, 8, k, 4)
a = torch.randn(1, 1, LEN); b = a.clone(); b[..., 50:] += 10.0        # change only the inputs from t = 50 on
print("outputs before t=50 unchanged after editing the future:", torch.allclose(enc(a)[..., :50], enc(b)[..., :50]))

fig = go.Figure(go.Heatmap(z=np.array(influence), x=np.arange(LEN), y=[f"L = {L}" for L in range(1, 6)],
                           colorscale=[[0, pf.INK["surface"]], [1, pf.BLUE[450]]], showscale=False, xgap=1, ygap=3,
                           hovertemplate="%{y}, lag %{x}: %{z:.0f} (1 = can influence today's output)<extra></extra>"))
fig.update_layout(title="Receptive field doubles with every residual block (k = 3)", height=300,
                  xaxis_title="lag (months before the forecast origin)", yaxis_title="", yaxis_autorange="reversed")
fig.show()

,dilations,formula R,measured R
blocks L,,,
1,[1],5,5
2,"[1, 2]",13,13
3,"[1, 2, 4]",29,29
4,"[1, 2, 4, 8]",61,61
5,"[1, 2, 4, 8, 16]",125,125


outputs before t=50 unchanged after editing the future: True


The measured receptive field matches the formula exactly, and the TCN is causal: editing inputs at $t\ge 50$ leaves
every output before $t=50$ unchanged. With $L=4$ blocks the network sees 61 months, more than the 36-month context we
will feed it.

## Part B: a panel of 300 SKUs
`pf.make_sku_panel()` generates 300 monthly count series (2015–2024) that resemble a real assortment:
* four product **families** with different, *non-sinusoidal* seasonal shapes (holiday spike, summer, back-to-school, spring),
* **scales** from under 1 to several hundred units per month, so slow movers are intermittent,
* persistent **demand shocks** (a latent AR(1) on the log rate), NegBin noise with SKU-specific over-dispersion,
* about 40% of SKUs **launch** during 2016–2022 with a ramp-up, so many have short histories.

Because the data are simulated, we also know the true rate $\mu_t$, which gives an unattainable **oracle** benchmark.

In [4]:
y_df, mu_df, info = pf.make_sku_panel()
Y = y_df.to_numpy(); T, N = Y.shape
first_sale = np.argmax(Y > 0, axis=0)
print(f"{N} SKUs x {T} months   share of zero months: {np.mean(Y == 0):.0%}")
print("families:", info.family.value_counts().to_dict())
print(f"SKUs launched after Jan 2015: {np.mean(info.launch > y_df.index[0]):.0%}")

show = {"holiday, fast mover": info.index[(info.family == "holiday") & (info.level > 60)][0],
        "summer, mid volume": info.index[(info.family == "summer") & info.level.between(8, 20) & (first_sale == 0)][0],
        "school, launched 2020": info.index[(info.family == "school") & (info.launch.dt.year == 2020)][0],
        "spring, slow mover": info.index[(info.family == "spring") & (info.level < 1.5)][0]}
fig = pf.subplots(2, 2, titles=[f"{k} ({v})" for k, v in show.items()], height=520, vertical_spacing=0.16)
for j, sku in enumerate(show.values()):
    fig.add_trace(go.Bar(x=y_df.index, y=y_df[sku], marker_color=pf.SERIES[0], showlegend=False, name=sku,
                         hovertemplate="%{x|%b %Y}: %{y:.0f} units<extra></extra>"), row=j // 2 + 1, col=j % 2 + 1)
fig.update_layout(title="Four of the 300 SKUs: shared seasonal families, very different scales and ages", bargap=0.15)
fig.update_yaxes(title_text="units", col=1)
fig.show()

300 SKUs x 120 months   share of zero months: 25%
families: {'holiday': 87, 'summer': 81, 'school': 66, 'spring': 66}
SKUs launched after Jan 2015: 39%


### Backtest design
| | months | used for |
|---|---|---|
| training windows | targets before **Jan 2022** (row 84) | fitting the network |
| validation origin | Jan 2022 → Dec 2022 | monitoring the loss, and later conformal calibration |
| **test origins** | **Jan 2023** and **Jan 2024**, 12 months each | all scores below (2 × 300 × 12 forecasts) |

The baselines are refitted at each test origin with all data before it. The network is not refitted. It just reads
the latest 36 months.

In [5]:
C, H = 36, 12                                         # context length, forecast horizon
TAUS = np.round(np.arange(0.05, 0.96, 0.05), 2)       # 19 quantile levels
K = len(TAUS)
VAL_ORIGIN, TEST_ORIGINS = 84, [96, 108]
month = y_df.index.month.to_numpy() - 1

def make_window(o):
    """Inputs (N, 4, C), scale nu (N,), scaled targets (N, H) for forecast origin o (first forecast month)."""
    ctx = Y[o - C:o].T
    nu = 1.0 + ctx.mean(axis=1)
    age = np.clip(np.arange(o - C, o)[None, :] - first_sale[:, None] + 1, 0, None)
    m = month[o - C:o]
    feats = np.stack([ctx / nu[:, None],
                      np.broadcast_to(np.sin(2 * np.pi * m / 12), (N, C)),
                      np.broadcast_to(np.cos(2 * np.pi * m / 12), (N, C)),
                      np.log1p(age) / np.log(T)], axis=1)
    tgt = Y[o:o + H].T / nu[:, None] if o + H <= T else np.full((N, H), np.nan)
    return feats.astype(np.float32), nu.astype(np.float32), tgt.astype(np.float32)

def stack_windows(origins):
    X, NU, TG = (np.concatenate(a) for a in zip(*[make_window(o) for o in origins]))
    keep = X[:, 0].sum(axis=1) > 0                    # drop windows of not-yet-launched SKUs (all-zero context)
    return torch.tensor(X[keep]), torch.tensor(NU[keep]), torch.tensor(TG[keep])

X_tr, NU_tr, T_tr = stack_windows(range(C, VAL_ORIGIN - H + 1))   # every origin whose targets end before row 84
X_va, NU_va, T_va = stack_windows([VAL_ORIGIN])
print(f"training windows: {len(X_tr):,}   validation windows: {len(X_va)}   input shape per window: {tuple(X_tr.shape[1:])}")

training windows: 9,142   validation windows: 298   input shape per window: (4, 36)


## Part C: the model and its training
Four residual blocks of 32 channels ($R=61$). The encoding of the **last** time step, together with $\log\nu$, goes
into a small MLP that outputs $12\times19$ non-crossing quantiles. The loss is the pinball loss averaged over all
horizons and quantile levels, on the scaled targets.

In [6]:
class QuantileTCN(nn.Module):
    def __init__(self, c_in=4, channels=32, k=3, n_blocks=4, dropout=0.1):
        super().__init__()
        self.encoder = tcn_encoder(c_in, channels, k, n_blocks, dropout)
        self.head = nn.Sequential(nn.Linear(channels + 1, 64), nn.ReLU(), nn.Linear(64, H * K))

    def forward(self, x, nu):
        h_last = self.encoder(x)[..., -1]                                  # (B, channels)
        a = self.head(torch.cat([h_last, torch.log(nu)[:, None]], dim=1)).view(-1, H, K)
        return torch.cat([a[..., :1], F.softplus(a[..., 1:])], dim=-1).cumsum(dim=-1)   # monotone in tau

TAU_T = torch.tensor(TAUS, dtype=torch.float32)
def pinball(q, target):
    u = target[..., None] - q
    return torch.maximum(TAU_T * u, (TAU_T - 1) * u).mean()

torch.manual_seed(13)                                  # training is reproducible regardless of the cells above
model = QuantileTCN()
print(f"receptive field R = {receptive_field(3, 4)} months, parameters = {sum(p.numel() for p in model.parameters()):,}")

receptive field R = 61 months, parameters = 39,556


In [7]:
EPOCHS, BATCH = 25, 256
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
hist, t0 = [], time.time()
for ep in range(EPOCHS):
    model.train()
    perm, tot = torch.randperm(len(X_tr)), 0.0
    for b in range(0, len(perm), BATCH):
        i = perm[b:b + BATCH]
        loss = pinball(model(X_tr[i], NU_tr[i]), T_tr[i])
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(i)
    model.eval()
    with torch.no_grad():
        hist.append({"epoch": ep + 1, "training": tot / len(X_tr), "validation": pinball(model(X_va, NU_va), T_va).item()})
print(f"trained {EPOCHS} epochs in {time.time() - t0:.0f} s")
hist = pd.DataFrame(hist).set_index("epoch")
fig = pf.add_lines_by(go.Figure(), hist, {"training": pf.SERIES[0], "validation": pf.SERIES[1]}, fmt=".4f", markers=False)
fig.update_layout(title="Pinball loss on scaled targets", height=340, xaxis_title="epoch", yaxis_title="mean pinball loss",
                  hovermode="x unified")
fig.show()

trained 25 epochs in 89 s


The validation loss is *lower* than the training loss: the training loss is computed with dropout switched on and
over many windows (including early, noisier launch periods), so the two curves are not directly comparable. What
matters is that the validation curve has flattened, so more epochs would not help much.

## Part D: forecasts and scores
Four forecasters at the two test origins:
* **Seasonal naive + empirical errors** (local): $y_{t-12}$ plus quantiles of the past seasonal-naive errors.
* **Local NegBin GLM**: per SKU, a Poisson GLM with trend + annual sine/cosine, NegBin dispersion by moments, fitted
  from the SKU's first sale (`pf.local_negbin_quantiles`).
* **TCN** (global): the network above, with quantiles clipped at 0 (demand cannot be negative).
* **Oracle**: the true NegBin($\mu_t$, $\alpha$) from the simulator. Not achievable (it knows the latent shocks); it shows the floor.

Scores (lower is better, all from `pf.panel_quantile_scores`):
* **wQL**, the volume-weighted quantile loss: big SKUs dominate it, as in revenue terms.
* **sQL**, the series-weighted loss: each SKU's pinball loss divided by its mean demand, so every SKU counts equally.
* **Coverage** of the central 80% and 90% intervals.

In [8]:
from scipy import stats

def tcn_quantiles(o):
    X, NU, _ = make_window(o)
    with torch.no_grad():
        q = model.eval()(torch.tensor(X), torch.tensor(NU)).numpy() * NU[:, None, None]
    return np.maximum(q, 0)                         # demand cannot be negative

def oracle_quantiles(o):
    m = mu_df.to_numpy()[o:o + H].T; n = 1 / info.alpha.to_numpy()[:, None]
    return stats.nbinom.ppf(TAUS[None, None, :], n[..., None], (n / (n + m))[..., None])

FORECASTERS = {"Seasonal naive": lambda o: pf.seasonal_naive_quantiles(Y, o, H, TAUS),
               "Local NegBin GLM": lambda o: pf.local_negbin_quantiles(Y, o, H, TAUS),
               "TCN (global)": tcn_quantiles,
               "Oracle (true μ)": oracle_quantiles}
COLOR = {"Seasonal naive": pf.SERIES[3], "Local NegBin GLM": pf.SERIES[1], "TCN (global)": pf.SERIES[0],
         "Oracle (true μ)": pf.INK["muted"]}

Q = {name: np.concatenate([f(o) for o in TEST_ORIGINS]) for name, f in FORECASTERS.items()}   # (2N, H, K) each
Y_test = np.concatenate([Y[o:o + H].T for o in TEST_ORIGINS])                                # (2N, H)
scale = np.concatenate([[Y[first_sale[i]:o, i].mean() + 1 for i in range(N)] for o in TEST_ORIGINS])
age = np.concatenate([o - first_sale for o in TEST_ORIGINS])
short = age < 36

def score_table(Q, mask=slice(None)):
    return pd.DataFrame({n: pf.panel_quantile_scores(Y_test[mask], q[mask], TAUS, scale[mask]) for n, q in Q.items()}).T

display(score_table(Q))
by_age = pd.DataFrame({f"sQL, history < 36 months (n={short.sum()})": score_table(Q, short)["sQL"],
                       f"sQL, history ≥ 36 months (n={(~short).sum()})": score_table(Q, ~short)["sQL"]})
by_age

,wQL,sQL,cov80,cov90
Seasonal naive,0.555,0.550,0.705,0.819
Local NegBin GLM,0.427,0.435,0.811,0.897
TCN (global),0.402,0.405,0.772,0.866
Oracle (true μ),0.333,0.340,0.853,0.932


,"sQL, history < 36 months (n=56)","sQL, history ≥ 36 months (n=544)"
Seasonal naive,0.720,0.532
Local NegBin GLM,0.771,0.400
TCN (global),0.461,0.399
Oracle (true μ),0.397,0.334


**Reading the scores**
* The global TCN beats both local baselines overall. The gain is concentrated in the **short-history SKUs**: a local
  GLM fitted to a year or two of a launch ramp extrapolates badly, while the TCN has seen hundreds of launches and
  seasonal cycles in other SKUs and transfers that knowledge.
* On SKUs with long histories, the local NegBin GLM is about as good as the TCN, because it is close to the true
  data-generating process. Deep learning wins where pooling information helps, not everywhere.
* The oracle's floor shows how much irreducible noise is left: NegBin noise on counts is large.
* The discrete forecasters (NegBin GLM, oracle) slightly **over**-cover, because integer quantiles include the whole
  probability atom at the interval ends. The TCN's intervals **under**-cover; Part E explains why and fixes it.

In [9]:
per_h = pd.DataFrame({n: [pf.panel_quantile_scores(Y_test[:, [h]], q[:, [h]], TAUS, scale)["wQL"] for h in range(H)]
                      for n, q in Q.items()}, index=pd.Index(range(1, H + 1), name="horizon (months)"))
fig = pf.add_lines_by(go.Figure(), per_h, COLOR, fmt=".3f")
fig.update_layout(title="wQL by forecast horizon (two test origins pooled)", height=380, yaxis_title="wQL",
                  xaxis_title="horizon (months)", hovermode="x unified")
fig.show()

The TCN's advantage is clearest at $h=1$: recent demand shocks (the latent AR(1) in the simulator) persist for a few
months, and the network reads them from the context, while the GLM only has a smooth trend + season. Further out the
shocks have decayed, the two models are close, and the month-to-month ranking is mostly noise (each point is an
average over 600 forecasts of very noisy counts).

In [10]:
o, pos = TEST_ORIGINS[1], 1                       # show the Jan 2024 origin (second block of rows)
idx = y_df.index[o:o + H]
fan_taus = [0.05, 0.25, 0.5, 0.75, 0.95]
names = list(show.items())[:3]
fig = pf.subplots(1, 3, titles=[k for k, _ in names], height=420)
for c, (label, sku) in enumerate(names, start=1):
    i = pos * N + y_df.columns.get_loc(sku)
    qd = {tau: Q["TCN (global)"][i, :, list(TAUS).index(tau)] for tau in fan_taus}
    pf.add_fan(fig, idx, qd, history=y_df[sku].iloc[:o], actual=Y_test[i], n_history=24, label="TCN",
               row=1, col=c, showlegend=c == 1, fmt=",.0f")
    fig.add_trace(go.Scatter(x=idx, y=Q["Local NegBin GLM"][i, :, list(TAUS).index(0.5)], mode="lines",
                             name="local GLM median", legendgroup="glm", showlegend=c == 1,
                             line=dict(color=COLOR["Local NegBin GLM"], width=2, dash="dash"),
                             hovertemplate="local GLM median: %{y:,.0f}<extra></extra>"), row=1, col=c)
fig.update_layout(title="TCN 50% / 90% intervals for 2024 (dashed: local GLM median)",
                  legend=dict(orientation="h", y=-0.15), margin=dict(b=90))
fig.update_yaxes(title_text="units", col=1)
fig.show()

Look at the holiday fast mover: the local GLM's median (dashed) climbs towards the December peak, the TCN's barely
moves. Is that one SKU, or systematic? Compare the **December / June ratio** of the median forecasts for all holiday
SKUs with the truth.

In [11]:
hol = np.tile((info.family == "holiday").to_numpy(), len(TEST_ORIGINS))
k50 = list(TAUS).index(0.5)
true_mu = np.concatenate([mu_df.to_numpy()[o:o + H].T for o in TEST_ORIGINS])
ratio = lambda m: np.median(m[hol, 11] / np.maximum(m[hol, 5], 0.5))      # Dec (h=12) / Jun (h=6)
pd.Series({"true μ": ratio(true_mu), "Local NegBin GLM (median)": ratio(Q["Local NegBin GLM"][..., k50]),
           "TCN (median)": ratio(Q["TCN (global)"][..., k50])}, name="median Dec/Jun ratio, holiday SKUs").round(2)

true μ                       2.53
Local NegBin GLM (median)    2.00
TCN (median)                 1.66
Name: median Dec/Jun ratio, holiday SKUs, dtype: float64

The TCN systematically **flattens the seasonal peak**: it puts the holiday SKUs' December only modestly above June,
while the truth is well over twice as high. A pinball loss on very noisy counts rewards hedging towards the typical
level, and a peak that occurs once a year is hard to learn from the last hidden state alone. The GLM has an explicit
seasonal term and gets closer. (The 850 units in Dec 2024 for the fast mover is far outside both models' intervals:
a 90% interval misses one month in ten, and big misses cluster in peak months.)

Typical remedies: longer training or a larger network, giving the model each SKU's **same-month-last-year** value
explicitly (a "seasonal lag" feature), or a hybrid that lets a simple seasonal model handle the calendar and the
network the rest.

## Part E: calibration, and a conformal fix
A quantile forecast is calibrated if $P(Y\le q_\tau)=\tau$ for every $\tau$. Plot the observed frequency against $\tau$.

Count data need one extra step. The TCN outputs *fractional* quantiles, but the quantiles of a count distribution are
integers. If the 10% quantile is 0.4, a month with zero sales falls *outside* the interval, although for a count
variable "0.4" really means "0". So we also plot the TCN with its quantiles **rounded** to integers.

In [12]:
Q["TCN, rounded"] = np.round(np.maximum(Q["TCN (global)"], 0))
COLOR["TCN, rounded"] = pf.SERIES[2]
calib = pd.DataFrame({n: [np.mean(Y_test <= q[..., k]) for k in range(K)] for n, q in Q.items()},
                     index=pd.Index(TAUS, name="τ"))
fig = go.Figure(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect calibration", hoverinfo="skip",
                           line=dict(color=pf.INK["muted"], dash="dash", width=1)))
pf.add_lines_by(fig, calib, COLOR, fmt=".3f")
fig.update_layout(title="Quantile calibration: share of actuals ≤ the τ-quantile", height=420, hovermode="x unified",
                  xaxis=dict(title="nominal level τ", range=[0, 1]), yaxis=dict(title="observed frequency", range=[0, 1]))
fig.show()

For counts, $P(Y\le q_\tau)$ sits **above** the diagonal even for a perfect model (the oracle), because $Y\le q$
includes the probability atom at $q$ (e.g. all the zero months). The **raw** TCN curve lies well below the oracle's.
Rounding closes about half of that gap (the fractional-quantile effect). The rest is genuine: the rounded TCN is still a
little below the oracle in the middle, i.e. its central quantiles are slightly too high. For the *intervals* this
matters less. Discrete intervals should cover *at least* their nominal level, and the rounded TCN's do (see the table below).

The general tool for genuine mis-calibration is **conformalised quantile regression (CQR, nb 04)**, adapted to a global model: on the validation origin (Jan 2022,
never used as a training target), compute the conformity score of every central interval $[q_\tau, q_{1-\tau}]$,
**per horizon** and **relative to the scale** $\nu$ so that big and small SKUs are comparable:

$$E_{i,h}=\max\big(q_{\tau}-y,\ y-q_{1-\tau}\big)/\nu_i,$$

take its $\lceil(n+1)(1-2\tau)\rceil/n$ empirical quantile $\hat Q_h$, and widen (or narrow) every test interval by
$\hat Q_h\,\nu_i$ on both sides. A final sort along $\tau$ restores monotonicity (rearrangement, nb 02), and we
round to integers again.

In [13]:
Xv, NUv, _ = make_window(VAL_ORIGIN)
with torch.no_grad():
    Qv = model.eval()(torch.tensor(Xv), torch.tensor(NUv)).numpy() * NUv[:, None, None]
Yv = Y[VAL_ORIGIN:VAL_ORIGIN + H].T
ok = Y[VAL_ORIGIN - C:VAL_ORIGIN].sum(axis=0) > 0                 # launched SKUs only
Qv, Yv, NUv = np.round(np.maximum(Qv[ok], 0)), Yv[ok], NUv[ok]    # calibrate the ROUNDED quantiles
n_cal = len(Yv)

nu_test = np.concatenate([make_window(o)[1] for o in TEST_ORIGINS])
Q_cqr = Q["TCN, rounded"].copy()
for k, tau in enumerate(TAUS):
    if tau >= 0.5:
        continue
    k_hi = list(TAUS).index(round(1 - tau, 2))
    E = np.maximum(Qv[..., k] - Yv, Yv - Qv[..., k_hi]) / NUv[:, None]          # (n_cal, H)
    level = min(np.ceil((n_cal + 1) * (1 - 2 * tau)) / n_cal, 1.0)
    Qh = np.quantile(E, level, axis=0, method="higher")                          # one correction per horizon
    Q_cqr[..., k] -= Qh[None, :] * nu_test[:, None]
    Q_cqr[..., k_hi] += Qh[None, :] * nu_test[:, None]
Q_cqr = np.round(np.maximum(np.sort(Q_cqr, axis=-1), 0))                         # rearrange, demand ≥ 0, integers
Q["TCN, rounded + per-horizon CQR"] = Q_cqr
score_table({n: Q[n] for n in ["Local NegBin GLM", "Oracle (true μ)", "TCN (global)", "TCN, rounded", "TCN, rounded + per-horizon CQR"]})

,wQL,sQL,cov80,cov90
Local NegBin GLM,0.427,0.435,0.811,0.897
Oracle (true μ),0.333,0.340,0.853,0.932
TCN (global),0.402,0.405,0.772,0.866
"TCN, rounded",0.402,0.403,0.824,0.900
"TCN, rounded + per-horizon CQR",0.401,0.403,0.835,0.913


* **Rounding** alone moves the TCN from under-coverage to (slightly above) nominal coverage, at no cost in wQL/sQL. For
  count targets, always report integer quantiles.
* After rounding, **CQR** only has a little left to fix and widens the intervals slightly. Its value is the
  finite-sample guarantee: with a model that is genuinely over-confident (try fewer epochs, or a TCN without dropout),
  CQR does the heavy lifting.
* General recipe for any neural quantile forecaster: train for sharpness, then **calibrate on held-out windows**.

## Takeaways
* A TCN is a stack of **causal, dilated convolutions** with residual connections. Its memory is exactly the receptive
  field $R=1+2(k-1)(2^L-1)$, and it trains in parallel.
* It becomes a probabilistic forecaster through its **head**. A direct multi-horizon quantile head with positive
  increments is simple, fast, and cannot cross.
* Deep forecasters are **global**: windows from many series, per-window scaling, calendar and age covariates. Their
  advantage is largest for **short histories** and **short horizons**; on long, clean series a good local model is a
  tough benchmark.
* Check calibration. For counts, round quantiles to integers first; then use **conformal** calibration on a held-out
  origin to fix what is left.
* Libraries: Darts `TCNModel(likelihood=QuantileRegression(...))`, Nixtla `neuralforecast.models.TCN`, GluonTS (MQ-CNN).

### Exercises
1. Set `n_blocks=2` ($R=13$). The network can no longer see last year's same month. What happens to the scores at $h=12$?
2. Replace the quantile head by a NegBin head (output $\mu,\alpha$ per horizon, minimise the NegBin negative
   log-likelihood from nb 14) and compare calibration on the slow movers.
3. MC dropout: keep `model.train()` at prediction time and run the network 100 times. How wide is the spread of the
   median forecast (epistemic uncertainty) compared with the 90% interval (total uncertainty)?
4. Retrain with 3 different seeds and average the quantiles (a deep ensemble, nb 05). Does wQL improve?
5. Add a seasonal-lag feature: for each target month, the SKU's value 12 months earlier (scaled by $\nu$), passed to
   the head. Does the December/June ratio for holiday SKUs get closer to the truth? (Simply passing the calendar month
   of each target step did *not* help in our tests.)